# Model Deployment - Practical Examples
## Learn how to save, load, and deploy models

## Part 1: Save and Load Models

In [ ]:
import pickle
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

print("=== MODEL SAVING AND LOADING ===")
print()

# Create and train a simple model
X, y = make_classification(n_samples=1000, n_features=20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Model trained with accuracy: {accuracy:.2%}")
print()

# Method 1: Pickle (simple but Python-only)
print("Method 1: Using Pickle")
with open('model_pickle.pkl', 'wb') as f:
    pickle.dump(model, f)
print("✓ Model saved using pickle")

with open('model_pickle.pkl', 'rb') as f:
    loaded_model_1 = pickle.load(f)
print("✓ Model loaded using pickle")
print(f"  Accuracy: {loaded_model_1.score(X_test, y_test):.2%}")
print()

# Method 2: Joblib (faster, better for large models)
print("Method 2: Using Joblib (PREFERRED)")
joblib.dump(model, 'model_joblib.joblib')
print("✓ Model saved using joblib")

loaded_model_2 = joblib.load('model_joblib.joblib')
print("✓ Model loaded using joblib")
print(f"  Accuracy: {loaded_model_2.score(X_test, y_test):.2%}")
print()

print("Comparison:")
print("Pickle: Simple, smaller files, Python-only")
print("Joblib: Faster, better for large arrays, Python-only")

## Part 2: Creating a Simple API with Flask

In [ ]:
print("\n=== CREATING FLASK API ===")
print()

# Code example for Flask API
flask_code = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

# Load model at startup
model = joblib.load('model_joblib.joblib')

@app.route('/predict', methods=['POST'])
def predict():
    """
    API endpoint for making predictions
    Input: JSON with 'features' key containing list of 20 numbers
    Output: JSON with prediction and probability
    """
    try:
        # Get input data
        data = request.json
        features = np.array(data['features']).reshape(1, -1)
        
        # Make prediction
        prediction = model.predict(features)[0]
        probability = model.predict_proba(features).max()
        
        # Return result
        return jsonify({
            'prediction': int(prediction),
            'probability': float(probability),
            'status': 'success'
        })
    
    except Exception as e:
        return jsonify({
            'error': str(e),
            'status': 'failed'
        }), 400

if __name__ == '__main__':
    app.run(debug=True, port=5000)
'''

print("Flask API Code:")
print(flask_code)
print()
print("To use this API:")
print("1. Install Flask: pip install flask")
print("2. Save code as app.py")
print("3. Run: python app.py")
print("4. API runs at http://localhost:5000")

In [ ]:
print("\n=== API USAGE EXAMPLE ===")
print()

print("Making a prediction via API:")
print()
print("curl -X POST http://localhost:5000/predict \\")
print("  -H 'Content-Type: application/json' \\")
print("  -d '{")
print('    "features": [1.2, -0.5, 0.3, 2.1, -1.5, 0.8, 1.1, -0.2, 0.5, 1.3,'
print("    0.2, -0.8, 1.0, 2.0, -0.3, 0.7, 1.5, -0.9, 0.4, 1.8]")
print("  }'")
print()
print("Response:")
print("""
{
  "prediction": 1,
  "probability": 0.92,
  "status": "success"
}
""")

## Part 3: Docker Containerization

In [ ]:
print("\n=== DOCKER CONTAINERIZATION ===")
print()

dockerfile = '''
FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY app.py .
COPY model_joblib.joblib .

EXPOSE 5000

CMD ["python", "app.py"]
'''

requirements = '''
flask==2.0.1
scikit-learn==0.24.2
joblib==1.0.1
numpy==1.21.0
pandas==1.3.0
'''

print("Dockerfile:")
print(dockerfile)
print()
print("requirements.txt:")
print(requirements)
print()
print("Docker commands:")
print("1. Build image: docker build -t spam-detector:latest .")
print("2. Run container: docker run -p 5000:5000 spam-detector:latest")
print("3. Push to Docker Hub: docker push username/spam-detector:latest")

## Part 4: Deployment Strategies

In [ ]:
print("\n=== DEPLOYMENT STRATEGIES ===")
print()

strategies = {
    'Blue-Green Deployment': {
        'Description': 'Run old and new version simultaneously',
        'Steps': [
            '1. New version runs in parallel (Green)',
            '2. Test new version thoroughly',
            '3. Switch traffic from old (Blue) to new (Green)',
            '4. Keep old version as fallback'
        ],
        'Advantage': 'Zero downtime, instant rollback',
        'Risk': 'Low (can rollback instantly)'
    },
    'Canary Deployment': {
        'Description': 'Gradually shift traffic to new version',
        'Steps': [
            '1. Deploy new version to small subset (5% traffic)',
            '2. Monitor performance',
            '3. If good, increase to 25%',
            '4. If good, increase to 50%',
            '5. Full rollout to 100%'
        ],
        'Advantage': 'Early issue detection',
        'Risk': 'Low (can roll back at any stage)'
    },
    'Rolling Deployment': {
        'Description': 'Update instances one by one',
        'Steps': [
            '1. Stop instance 1, deploy new version',
            '2. Start instance 1',
            '3. Stop instance 2, deploy new version',
            '4. Continue until all updated'
        ],
        'Advantage': 'Always has running version',
        'Risk': 'Medium (some instances may have issues)'
    }
}

for strategy, details in strategies.items():
    print(f"\n{strategy}")
    print("-" * 50)
    print(f"Description: {details['Description']}")
    print(f"Steps:")
    for step in details['Steps']:
        print(f"  {step}")
    print(f"Advantage: {details['Advantage']}")
    print(f"Risk: {details['Risk']}")

## Part 5: Complete Deployment Checklist

In [ ]:
print("\n=== DEPLOYMENT CHECKLIST ===")
print()

checklist = [
    ('Model Saving', 'Save model in production format (.joblib)'),
    ('API Creation', 'Create API endpoint with Flask/FastAPI'),
    ('Error Handling', 'Handle errors gracefully'),
    ('Logging', 'Log all predictions and errors'),
    ('Input Validation', 'Validate input before prediction'),
    ('Testing', 'Unit tests and integration tests pass'),
    ('Containerization', 'Docker image created and tested'),
    ('Documentation', 'README and API docs complete'),
    ('Monitoring Setup', 'Alerts configured for failures'),
    ('Backup Plan', 'Previous model saved for rollback'),
    ('Performance Baseline', 'Baseline metrics established'),
    ('Team Review', 'Code review completed'),
]

for task, description in checklist:
    print(f"[ ] {task:20} - {description}")

print()
print("Once all items checked: Ready for deployment!")

## Summary

1. **Model Saving** - Use joblib for sklearn models
2. **API Creation** - Flask/FastAPI makes models accessible
3. **Containerization** - Docker ensures consistency
4. **Deployment Strategies** - Choose based on risk tolerance
5. **Monitoring** - Essential after deployment

Next: Learn about Experiment Tracking in Section 4!